# Notebook 08 — Training monitor

Quick sanity checks to run BEFORE and DURING training.
No GPU required for most cells.

**Before training:**
- Check initial loss is ~10.82 (= ln(50257), the expected random baseline)
- Overfit a single batch (loss should go to ~0 in ~50 steps)
- Estimate GPU memory requirements

**During / after training:**
- Plot the loss curve from a checkpoint log file
- Check the model can complete a simple prompt

In [ ]:
# ═══════════════════════════════════════════════════════════
# CONFIGURATION
# ═══════════════════════════════════════════════════════════

CHECKPOINT_PATH = "../teaching_assistant/checkpoints/step_005000.pt"
LOG_FILE        = None   # optional: path to a text log file from train.py

# Model size to test (should match your train.py config)
N_LAYER = 6
N_HEAD  = 8
N_EMBD  = 512
BLOCK_SIZE = 512

In [ ]:
# ═══════════════════════════════════════════════════════════
# SANITY CHECK 1: does a freshly initialized model have
# loss ≈ ln(50257) ≈ 10.82?
#
# If it doesn't, something is wrong with the model init,
# the loss function, or the label masking.
# ═══════════════════════════════════════════════════════════
import sys, math, torch
sys.path.insert(0, "../teaching_assistant")
from model import TeachingAssistantModel, TransformerConfig

device = "cuda" if torch.cuda.is_available() else "cpu"
cfg    = TransformerConfig(n_layer=N_LAYER, n_head=N_HEAD, n_embd=N_EMBD,
                            block_size=BLOCK_SIZE)

model = TeachingAssistantModel(cfg).to(device)
model.eval()

# Random batch
B, T = 4, 64
x = torch.randint(0, cfg.vocab_size, (B, T)).to(device)
with torch.no_grad():
    _, loss = model(x, x)

expected = math.log(cfg.vocab_size)
print(f"Initial loss:  {loss.item():.4f}")
print(f"Expected ~:    {expected:.4f}  (= ln({cfg.vocab_size}))")
print(f"Difference:    {abs(loss.item() - expected):.4f}")
print()
if abs(loss.item() - expected) < 0.5:
    print("✓ Initialization looks correct")
else:
    print("✗ Unexpected initial loss — check model init and loss function")

In [ ]:
# ═══════════════════════════════════════════════════════════
# SANITY CHECK 2: can we overfit a single batch?
#
# Train on the same 4-sequence batch for 50 steps.
# Loss should decrease consistently to near 0.
# If it plateaus or diverges, there's a bug in the training loop.
# ═══════════════════════════════════════════════════════════
import tiktoken

enc = tiktoken.get_encoding("gpt2")

# A tiny fixed training sequence
text = "Question: What is dropout?\nAnswer: Dropout randomly zeroes neuron outputs.<|endoftext|>"
ids  = enc.encode(text, allowed_special={"<|endoftext|>"})

# Repeat to fill a batch
ids_t = torch.tensor(ids[:64], dtype=torch.long)
x = ids_t.unsqueeze(0).expand(4, -1).to(device)

# Fresh model
model2 = TeachingAssistantModel(TransformerConfig(
    n_layer=2, n_head=2, n_embd=64, block_size=128  # tiny for speed
)).to(device)
model2.train()
opt = torch.optim.AdamW(model2.parameters(), lr=3e-4)

print("Overfitting single batch (50 steps):")
for step in range(50):
    opt.zero_grad()
    _, loss = model2(x, x)
    loss.backward()
    opt.step()
    if step % 10 == 0:
        print(f"  step {step:3d}  loss={loss.item():.4f}")

print()
if loss.item() < 1.0:
    print("✓ Successfully overfit — training loop works")
else:
    print("✗ Loss didn't go below 1.0 — check your training loop")

In [ ]:
# ═══════════════════════════════════════════════════════════
# MEMORY ESTIMATE
# How much GPU RAM will training require?
# Rule of thumb: ~6 bytes per parameter for training
# (2 bytes weights + 2 bytes gradients + 2 bytes Adam state)
# Plus activations (batch_size × seq_len × n_embd × n_layers × ~4 bytes)
# ═══════════════════════════════════════════════════════════
n_params = model.count_parameters()
param_memory_mb   = n_params * 4 / 1e6          # float32 weights
training_overhead = n_params * 6 / 1e6          # weights + gradients + Adam

# Activation memory (rough estimate for batch_size=8, seq_len=512)
batch_size, seq_len = 8, 512
activation_mb = batch_size * seq_len * N_EMBD * N_LAYER * 4 / 1e6

total_mb = training_overhead + activation_mb

print(f"Model parameters:    {n_params:>12,}")
print(f"Parameter memory:    {param_memory_mb:>10.0f} MB  (inference, float32)")
print(f"Training overhead:   {training_overhead:>10.0f} MB  (weights + grads + Adam)")
print(f"Activations:         {activation_mb:>10.0f} MB  (batch={batch_size}, seq={seq_len})")
print(f"─────────────────────────────")
print(f"Estimated total:     {total_mb:>10.0f} MB")
print()
if total_mb < 4000:
    print("✓ Fits on most laptop GPUs (4GB+)")
elif total_mb < 8000:
    print("✓ Fits on 3080 laptop (8GB) — your target hardware")
else:
    print("⚠ May be tight on 8GB VRAM — consider reducing batch_size or n_embd")

In [ ]:
# ═══════════════════════════════════════════════════════════
# LOAD A CHECKPOINT AND TEST GENERATION
# Check if a trained checkpoint can complete a QA prompt.
# ═══════════════════════════════════════════════════════════
from pathlib import Path

if not Path(CHECKPOINT_PATH).exists():
    print(f"No checkpoint at {CHECKPOINT_PATH} — run train.py first.")
else:
    from rag_pipeline import load_model
    trained_model, tokenizer, config = load_model(CHECKPOINT_PATH, device)

    test_prompts = [
        "Question: What is dropout?\nAnswer:",
        "Question: How does gradient descent work?\nAnswer:",
    ]

    for prompt in test_prompts:
        import tiktoken
        enc2 = tiktoken.get_encoding("gpt2")
        ids = enc2.encode(prompt)
        x   = torch.tensor([ids], dtype=torch.long).to(device)

        out = trained_model.generate(x, max_new_tokens=100,
                                      temperature=0.7, top_k=50,
                                      stop_token=enc2.eot_token)
        new_text = enc2.decode(out[0, len(ids):].tolist())
        new_text = new_text.replace("<|endoftext|>", "").strip()

        print(f"Prompt:   {prompt}")
        print(f"Response: {new_text[:200]}")
        print()

In [ ]:
# ═══════════════════════════════════════════════════════════
# PLOT LOSS CURVE (if you have a log file)
# Your train.py prints lines like:
#   step   100 | loss 4.3210 | ppl ...
# Save stdout to a file and point LOG_FILE at it.
# ═══════════════════════════════════════════════════════════
import re

if LOG_FILE and Path(LOG_FILE).exists():
    steps, losses = [], []
    with open(LOG_FILE) as f:
        for line in f:
            m = re.search(r'step\s+(\d+).*loss\s+([\d.]+)', line)
            if m:
                steps.append(int(m.group(1)))
                losses.append(float(m.group(2)))

    if steps:
        try:
            import matplotlib.pyplot as plt
            plt.figure(figsize=(10, 4))
            plt.plot(steps, losses, linewidth=1.5)
            plt.xlabel("Step")
            plt.ylabel("Loss")
            plt.title("Training loss curve")
            plt.grid(alpha=0.3)
            plt.tight_layout()
            plt.show()
            print(f"Final loss: {losses[-1]:.4f}")
        except ImportError:
            print("matplotlib not installed — printing loss values instead")
            for s, l in zip(steps[::max(1, len(steps)//20)], losses[::max(1, len(steps)//20)]):
                print(f"  step {s:6d}: {l:.4f}")
else:
    print("No log file specified — set LOG_FILE at the top of this notebook.")
    print("Example: python train.py 2>&1 | tee training_log.txt")